In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("..")
from modeling import COVARIATES, oof_predict, prediction_metrics

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
OUT_DIR = DATA_DIR / "train_diversity"
TARGETS = ["Richness", "Shannon_diversity"]


def read(path):
    df = pd.read_csv(path, low_memory=False)
    df["participant_id"] = df["participant_id"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    return df


diet = read(DATA_DIR / "diet_experiment" / "diet_feature_matrix_baseline_full.csv")
covariates = read(DATA_DIR / "covariates" / "covariates_age_sex_bmi_smoking_oralhealth_baseline.csv")
diversity = read(DATA_DIR / "oral_experiment" / "oral_diversity_baseline.csv")
diet_features = [c for c in diet.columns if c != "participant_id"]
model_input = (covariates[["participant_id"] + COVARIATES].merge(diet, on="participant_id")
               .merge(diversity[["participant_id"] + TARGETS], on="participant_id")
               .drop_duplicates("participant_id"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
model_input.to_csv(OUT_DIR / "model_input_diet_oral_diversity_baseline.csv", index=False)

In [ ]:
models = {"covariates_only": COVARIATES, "covariates_plus_diet": COVARIATES + diet_features}
rows = []
for target in TARGETS:
    y = model_input[target].to_numpy(dtype=float)
    for model, features in models.items():
        metrics = prediction_metrics(y, oof_predict(model_input[features], y))
        rows.append({"target": target, "model": model, "n": len(y), **metrics})
results = pd.DataFrame(rows)
results["delta_r"] = (results["pearson_r"] - results.groupby("target")["pearson_r"].transform("first")).where(results["model"] == "covariates_plus_diet")
results.to_csv(OUT_DIR / "model_results_diet_predict_oral_diversity_baseline.csv", index=False)
print(results)